# PhasePhyto Plot Viewer

Displays saved plot/image artifacts for one run: training curves, illumination invariance, confusion matrices, and sample analysis figures.


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    # Set this to a specific run folder name under drive_project_dir/runs.
    # If None, the latest run by folder name is used.
    "run_name": None,
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + LOCATE RUN
# ============================================================

from pathlib import Path
import json

from google.colab import drive
from IPython.display import Image, Markdown, display

drive.mount("/content/drive", force_remount=False)

PROJECT_DIR = Path(CONFIG["drive_project_dir"])
RUNS_DIR = PROJECT_DIR / "runs"

if not RUNS_DIR.exists():
    raise RuntimeError(f"No runs directory found: {RUNS_DIR}")

if CONFIG["run_name"]:
    RUN_DIR = RUNS_DIR / CONFIG["run_name"]
else:
    candidates = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No run directories found under {RUNS_DIR}")
    RUN_DIR = candidates[-1]

MANIFEST_PATH = RUN_DIR / "run_manifest.json"
if not MANIFEST_PATH.exists():
    raise RuntimeError(f"No run_manifest.json found at {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text())
artifacts = manifest.get("artifacts", {})

print(f"Loaded run: {RUN_DIR}")
print(f"Manifest: {MANIFEST_PATH}")


In [ ]:
# ============================================================
# DISPLAY MAIN PLOTS
# ============================================================

plot_keys = [
    "training_curves",
    "illumination_invariance",
    "confusion_matrices",
]

for key in plot_keys:
    path = Path(artifacts.get(key, ""))
    if path.exists():
        display(Markdown(f"## {key.replace('_', ' ').title()}"))
        display(Image(filename=str(path)))
    else:
        print(f"Missing plot for {key}: {path}")


In [ ]:
# ============================================================
# DISPLAY ANALYSIS SAMPLES
# ============================================================

sample_glob = artifacts.get("analysis_samples_glob", str(RUN_DIR / "plots" / "analysis_sample_*.png"))
if sample_glob.startswith("/"):
    sample_paths = sorted(Path("/").glob(sample_glob.lstrip("/")))
else:
    sample_paths = sorted(Path().glob(sample_glob))

if sample_paths:
    display(Markdown("## Analysis Samples"))
    for path in sample_paths[:12]:
        display(Markdown(f"### {path.name}"))
        display(Image(filename=str(path)))
else:
    print(f"No analysis sample images matched: {sample_glob}")
